In [1]:
import argparse, json, pathlib, sys
from typing import Any, List, Sequence, Union

Json = Union[dict, list, str, int, float, bool, None]
JsonPath = Sequence[Union[str, int]]

In [2]:
def search_json(
    node: Json,
    target: Any,
    contains: bool,
    path_so_far: Sequence[Union[str, int]],
    results: List[Sequence[Union[str, int]]],
) -> None:
    """Depth-first walk that records every matching path."""
    # Dict → recurse on each key/value
    if isinstance(node, dict):
        for key, value in node.items():
            search_json(value, target, contains, path_so_far + [key], results)

    # List → recurse on each element
    elif isinstance(node, list):
        for idx, value in enumerate(node):
            search_json(value, target, contains, path_so_far + [idx], results)

    # Primitive → compare
    else:
        match = False
        if contains and isinstance(node, str) and isinstance(target, str):
            match = target.lower() in node.lower()
        else:
            match = node == target

        if match:
            results.append(path_so_far)


def to_newtonsoft(
    path: JsonPath,
    *,
    escape_backslashes: bool = False,
    safe: bool = False,
) -> str:
    """
    Convert a `search_json` path to a Newtonsoft‑style accessor.

    Parameters
    ----------
    escape_backslashes : bool
        If True, every back‑slash is doubled *after* the key has been
        quote‑escaped, so the whole string can be pasted into a plain
        C# string literal.
    safe : bool
        If True, inserts the null‑conditional operator `?` in front of
        every '[' segment.
    """
    parts = ["root"]
    for seg in path:
        prefix = "?[" if safe else "["
        if isinstance(seg, int):
            parts.append(f"{prefix}{seg}]")
            continue

        # 1️⃣ start with the raw key text
        txt = seg
        # 2️⃣ escape double‑quotes first (this introduces back‑slashes)
        txt = txt.replace('"', r'\"')
        # 3️⃣ *then* optionally double every back‑slash
        if escape_backslashes:
            txt = txt.replace("\\", r"\\")
        parts.append(f'{prefix}"{txt}"]')
    return "".join(parts)


import re

import json
def print_newtonsoft(json_data, newtonsoft_path: str) -> None:
    """
    Evaluate a Newtonsoft-style path (with or without '?' safety operators)
    and pretty-print the value it resolves to.
    """

    # Strip optional '?' operators that come from `to_newtonsoft(..., safe=True)`
    newtonsoft_path = newtonsoft_path.replace("?", "")

    # Match either ["string key"] or [123] segments
    parts = re.findall(
        r'\[\s*"((?:[^"\\]|\\.)*)"\s*\]'   # group 1: string key
        r'|\[\s*(\d+)\s*\]',              # group 2: list index
        newtonsoft_path
    )

    current = json_data
    for key, idx in parts:
        if key:                           # dict access
            key = bytes(key, "utf-8").decode("unicode_escape")
            if not isinstance(current, dict) or key not in current:
                print(f"❌ Key '{key}' not found at this level.")
                return
            current = current[key]
        else:                             # list index access
            i = int(idx)
            if not isinstance(current, list) or i >= len(current):
                print(f"❌ Index [{i}] out of range.")
                return
            current = current[i]

    print("\n✅ Final value:")
    print(json.dumps(current, indent=2, ensure_ascii=False))




In [3]:
import json

# Load JSON data
with open("my.json", "r", encoding="utf-8") as f:
    data = json.load(f)


In [5]:
# Perform the search
target_value = "Stark Company Realtors"
matches = []
search_json(data, target_value, contains=True, path_so_far=[], results=matches)

# Display results as C# Newtonsoft paths

for match in matches:
    p1 = to_newtonsoft(match, escape_backslashes=False, safe=True)
    p2 = to_newtonsoft(match, escape_backslashes=True)
    print("C#: ", p1) 
    print("\nEscaped: ", p2) 
    print_newtonsoft(data, p1)
    print("\n\n")

C#:  root?["data"]?["presentation"]?["stayProductDetailPage"]?["sections"]?["sections"]?[8]?["section"]?["hostHighlights"]?[0]?["title"]

Escaped:  root["data"]["presentation"]["stayProductDetailPage"]["sections"]["sections"][8]["section"]["hostHighlights"][0]["title"]

✅ Final value:
"My work: Stark Company Realtors"





In [9]:
print_newtonsoft(data, 'root["data"]["presentation"]["stayProductDetailPage"]["sections"]["sections"][8]["section"]["hostHighlights"]')



✅ Final value:
[
  {
    "__typename": "HostHighlight",
    "icon": "SYSTEM_BRIEFCASE",
    "title": "My work: Stark Company Realtors"
  },
  {
    "__typename": "HostHighlight",
    "icon": "SYSTEM_GLOBE_STAND",
    "title": "Lives in Cambridge, Wisconsin"
  }
]


In [6]:
def get_utf8_size(obj):
    return len(json.dumps(obj, separators=(",", ":"), ensure_ascii=False).encode("utf-8"))

def traverse(obj, path="$", depth=0, max_depth=6, results=None):
    if depth > max_depth:
        return

    size = get_utf8_size(obj)
    if size > 20000:
        results.append((path, size, depth))

    if isinstance(obj, dict):
        for key, value in obj.items():
            traverse(value, f"{path}.{key}", depth + 1, max_depth, results)
    elif isinstance(obj, list):
        for index, item in enumerate(obj):
            traverse(item, f"{path}[{index}]", depth + 1, max_depth, results)

In [7]:
results = []
traverse(data, results=results)
results.sort(key=lambda x: x[1], reverse=True)

# Print sorted results
for path, size, depth in results:
    indent = "  " * depth
    print(f"{indent}{path} = {size} bytes")

$ = 112512 bytes
  $.ROOT_QUERY = 109504 bytes
    $.ROOT_QUERY.productGallery({"context":{"currency":"USD","device":{"type":"DESKTOP"},"eapid":1,"identity":{"authState":"ANONYMOUS","duaid":"83b9e1d3-6d20-bf36-dcfc-c05d5576d5cc"},"locale":"en_US","privacyTrackingState":"CAN_TRACK","siteId":9001001,"tpid":9001},"productIdentifier":{"id":"58948250","shoppingContext":{"multiItem":null,"queryTriggeredBy":"OTHER"},"travelSearchCriteria":{"property":{"primary":{"dateRange":null,"destination":{"coordinates":{"latitude":0,"longitude":0},"mapBounds":null,"pinnedPropertyId":null,"propertyIds":null,"regionId":"6023185","regionName":"Africa"},"rooms":[{"adults":2,"children":[]}]},"secondary":{"booleans":[],"counts":[],"ranges":[],"selections":[{"id":"privacyTrackingState","value":"CAN_TRACK"},{"id":"productOffersId","value":"8cc83548-2651-4929-85fc-e1fa0aee9b63"},{"id":"searchId","value":"14df1e36-bf04-457d-912a-cab3c773080e"},{"id":"sort","value":"RECOMMENDED"},{"id":"useRewards","value":"SHOP_WI

In [8]:
print_newtonsoft(data, 'root["data"]["presentation"]["stayProductDetailPage"]["sections"]["sections"][25]["section"]["seeAllAmenitiesGroups"]')

❌ Key 'data' not found at this level.
